In [ ]:
batch = next(iter(train_dataloader))

token_ids = batch["encoder_input"][0:1, :15]

print(token_ids.shape)

for tid in token_ids[0]:
    print(tid.item(), tokenizer_src.id_to_token(tid.item()))
    
x = encoder_input(token_ids)
x = position_encoder(x)

print(x.shape)

torch.Size([1, 15])
1 [SOS]
12 '
500 Is
17 it
382 possible
15 that
70 we
204 shall
37 be
96 like
285 husband
6 and
230 wife
4 ,
335 alone
torch.Size([1, 15, 512])


In [ ]:
from src.attention import SingleHeadAttention

In [ ]:
head = SingleHeadAttention(
    d_in=DIMENSION_SIZE, 
    d_out=64,
    dropout=0.0
)

In [ ]:
Q = head._w_query(x)
K = head._w_key(x)
V = head._w_value(x)

print("X:", x.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

X: torch.Size([1, 15, 512])
Q: torch.Size([1, 15, 64])
K: torch.Size([1, 15, 64])
V: torch.Size([1, 15, 64])


In [ ]:
token_idx = 5

print("Token:", tokenizer_src.id_to_token(token_ids[0, token_idx].item()))

print("\nQ:")
print(Q[0, token_idx, :])

print("\nK:")
print(K[0, token_idx, :])

print("\nV:")
print(V[0, token_idx, :])

Token: that

Q:
tensor([-18.8944,   0.2161,  -4.8120, -14.7146,  18.1875, -14.0109,  -8.0739,
        -26.5579,  23.9003, -21.8716, -10.5956,   1.2605, -14.9634,   5.5238,
        -12.3182,   1.6386,  -4.9557,  -2.5798,  -2.2219,  -4.4382,  13.2908,
        -13.5769,  -2.2165,   0.3443,   6.1813,  19.3228,   6.3108,  10.1801,
         -4.2095,  -9.6434,  -6.7741,  18.5230,  13.4252,  -2.6346,   3.8072,
          8.4598, -21.5483,  -6.6732,  -1.7801, -26.5174,  -2.7963,  -1.3902,
        -10.2941,  15.2010,  -4.4116,  -9.4893,  30.6937,  -3.1517, -13.5532,
          4.3264, -10.9181, -16.5706,   3.7082,  -1.3721,  14.7573,   5.7539,
          4.8301,   3.5585,  11.6996,  -2.3705, -14.4643,  -4.9688,  13.1876,
          8.3386], grad_fn=<SelectBackward0>)

K:
tensor([ -0.2015, -12.6046,  10.1486,  -5.5408,  20.3812, -19.9879,  -3.3853,
        -12.9545,  11.7209,  -1.0286,  32.9206,  28.1171,   3.7050,  -5.0219,
         16.6232,  12.3972,  12.7918,  13.1002,  -4.6917, -16.3358,   5.6246

In [ ]:
q1 = Q[0, 1]
k2 = K[0, 2]

score_1_2 = q1 @ k2

print(score_1_2)

tensor(405.9514, grad_fn=<DotBackward0>)


In [ ]:
scores_token1 = Q[0, 1] @ K[0].T

print(scores_token1.shape)
print(scores_token1)

torch.Size([15])
tensor([  889.6376,  1435.2827,   405.9510,  -318.0321, -2539.2437,   863.1581,
        -1043.5369,   223.6810,   756.6732,  -303.5441, -3500.5781,  -483.3849,
         2404.3657, -3514.5442, -2756.9443], grad_fn=<SqueezeBackward4>)


In [ ]:
scores = Q @ K.transpose(-2, -1)

print(scores.shape)
print(scores[0])

torch.Size([1, 15, 15])
tensor([[  949.6100,  -749.0314, -1266.3293, -2480.0134,  1076.0151, -2136.3936,
          -737.7044,    -7.9376,  -129.9576, -1247.8807,  3606.7659,   688.5898,
          -291.5875,  2241.5020,   401.6828],
        [  889.6379,  1435.2827,   405.9509,  -318.0323, -2539.2439,   863.1584,
         -1043.5370,   223.6810,   756.6730,  -303.5441, -3500.5781,  -483.3848,
          2404.3662, -3514.5442, -2756.9443],
        [-1033.3757, -1542.4071, -2519.1682,   706.6434,   154.4716, -1493.1324,
          3361.6902,  -964.0763,  -316.2359, -2316.2495,  1187.8478,  1130.2969,
          1151.8074,  1986.1416, -1970.9075],
        [  703.8950,  -439.5220,  -251.4643, -1455.5903,   280.9255,  5119.3926,
           -75.4530,  1983.8167,   503.0096,  -377.9642,  -624.7032,  -474.5362,
            34.6458,  1959.7499,   589.9705],
        [  -93.2373, -1317.3916,  1689.9092, -2170.4407,   632.1123,  2199.7393,
           450.7095,   392.7991,  -589.3851,  -881.1001, -2286.

In [ ]:
attention_scores = (
    Q @ K.transpose(-2, -1)
) / math.sqrt(64)
attention_weights = torch.softmax(
    attention_scores,
    dim=-1
)
print(attention_weights.shape)
print(attention_weights[0])
output = attention_weights @ V

print(output.shape)

torch.Size([1, 15, 15])
tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 2.1035e-28, 0.0000e+00, 0.0000e+00, 1.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000